In [ ]:
from markitdown import MarkItDown

def pdf_to_markdown(pdf_path, md_path):
    md = MarkItDown()
    # Extract and convert
    result = md.convert(pdf_path)
    
    # Save the result to a .md file
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(result.text_content)

# # Example usage
# pdf_file = "sample.pdf"
# md_file = "output.md"
# markdown_text = pdf_to_markdown(pdf_file, md_file)

In [5]:
from pathlib import Path
from typing import List,Any
import os
def load_all_documents(data_dir: str)-> List[Any]:
    data_path = Path(data_dir).resolve()
    pdf_files = list(data_path.glob("**/*.pdf"))
   # Define a directory path, not a file path
    md_output_directory = Path("../data/md")
    md_output_directory.mkdir(parents=True, exist_ok=True) # Ensure directory exists
    
    for pdf_file in pdf_files:
        # Changes "document.pdf" -> "document.md" inside the output folder
        md_file_name = pdf_file.stem + ".md"
        md_file_path = md_output_directory / md_file_name
        
        pdf_to_markdown(str(pdf_file), str(md_file_path))
        print(f"Converted: {pdf_file.name} -> {md_file_name}")

load_all_documents("../data/pdf")

Converted: gs statement uwa.pdf -> gs statement uwa.md
Converted: Sanjana_SOP_UWA_MIT_v4.pdf -> Sanjana_SOP_UWA_MIT_v4.md


In [ ]:
import os
from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from dotenv import load_dotenv

load_dotenv()
CHROMA_DB_DIR = "../data/chroma_db"

md_files = list(Path("../data/md").resolve().glob("**/*.md"))

splitter = MarkdownTextSplitter(chunk_size=1000, chunk_overlap=100)

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'}
)

all_chunks = []
for md_file in md_files:
    with open(str(md_file), "r", encoding="utf-8") as f:
        text = f.read()
    
    raw_chunks = splitter.split_text(text)
    for chunk in raw_chunks:
        doc = Document(
            page_content=chunk,
            metadata={"source": md_file.name, "path": str(md_file)}
        )
        all_chunks.append(doc)

print(f"Total chunks created across all files: {len(all_chunks)}")

if all_chunks:
    print("Embedding chunks and saving to disk... (this might take a moment)")
    vector_db = Chroma.from_documents(
        documents=all_chunks, 
        embedding=embedding_model, 
        persist_directory=CHROMA_DB_DIR
    )
    print(f"Success! Vector database saved to '{CHROMA_DB_DIR}'")
else:
    print("No markdown files found or processed. Database not created.")

e:\Personal Project\my_first_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1316.95it/s]


Total chunks created across all files: 25
Embedding chunks and saving to disk... (this might take a moment)
Success! Vector database saved to '../data/chroma_db'


In [5]:
# Load the existing database from disk
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings 
import os

load_dotenv()
CHROMA_DB_DIR = "../data/chroma_db"

# embedding_model = HuggingFaceInferenceAPIEmbeddings(
#     api_key=os.getenv("HUGGINGFACEHUB_API_TOKEN"), 
#     model_name="BAAI/bge-small-en-v1.5"
# )
vector_db = Chroma(
    persist_directory=CHROMA_DB_DIR, 
    embedding_function=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

# Search across all your documents
query = "Tell me about sanjana's sponsor"
results = vector_db.similarity_search(query, k=3)

context = [match.page_content for match in results]
# Print results with their source file
for i, match in enumerate(results):
    print(f"\n--- Match #{i+1} (From: {match.metadata['source']}) ---")
    print(match.page_content)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3509.58it/s]



--- Match #1 (From: gs statement uwa.md) ---
1.  I, Sanjana Akter Roshni, reside in Dhaka, Bangladesh, with my husband, who is

employed as a Software Engineer. My father owns and manages Sydney Homes Ltd, a

real estate company specializing in residential apartment construction and sales. I am not

currently involved in any community or NGO leadership roles.

My primary sponsor is my father, with fixed deposit of BDT 50,00,000 (approximately

AUD 57,145). My secondary sponsor is my mother-in-law, with fixed deposit of BDT

40,00,000 (approximately AUD 45,716), making a combined total of BDT 90,00,000

(approximately AUD 102,861). My father’s annual income is approximately BDT

35,50,000. He owns a real estate company and land valued at approximately AUD

123,609 while my mother-in-law owns property worth approximately AUD 427,014.

I have been working as a Programmer at Computer Ease Limited since 12 May 2024,

earning BDT 41,600 monthly. I have no study gaps and no military obligati

In [4]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)
prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
response=llm.invoke([prompt.format(context=context,query=query)])
print(response.content)

Hasibul Hoque is an established Software Engineer with over three years of experience specializing in Java backend systems, microservices, and enterprise banking software. He is married to the applicant and has been building their home and future together in Dhaka, Bangladesh.
